<a href="https://colab.research.google.com/github/thomasjwilloughby/LabModule5Part2/blob/main/Notebooks/AL01-Introduction/AL1_Image_Processing_Fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background:#1a3a5c;color:white;padding:20px 24px;border-radius:8px;margin-bottom:8px">
<h1 style="margin:0;font-size:1.8em">Active Learning AL1 — Introduction to Python Image Processing</h1>
<p style="margin:6px 0 0 0;opacity:0.85">MCS 3950 Computer Vision &nbsp;·&nbsp; UPEI Fall 2026</p>
</div>


## Overview

Welcome to your first Computer Vision Active Learning Lecture! Today you will set up your working environment,
learn how images are represented as NumPy arrays, and practice the basic operations you will
use throughout the entire course.

We'll be working with aerial and satellite imagery of **Prince Edward Island** wherever
possible — the same kinds of images used in real-world applications like coastal monitoring,
agricultural mapping, and 3D reconstruction.

---

### Learning Objectives
By the end of this session you will be able to:
1. Verify your Python / OpenCV / NumPy environment is correctly installed.
2. Load, display, and save images using OpenCV and Matplotlib.
3. Explain how a digital image is stored as a NumPy array (shape, dtype, pixel values).
4. Describe the difference between BGR (OpenCV) and RGB (Matplotlib/PIL) colour ordering.
5. Convert between colour spaces (RGB, Grayscale, HSV).
6. Perform basic spatial operations: crop, resize, flip, rotate.
7. Compute and interpret an image histogram.

---

### How to use this notebook
- **Read** each markdown cell before running the code below it.
- **Run** code cells with `Shift + Enter`.
- Cells labelled `# YOUR CODE HERE` are exercises — **do not skip them**.

> 💡 **Tip:** If a cell raises an `ImportError`, jump to Part 0 and re-run the install cell.


<div style="background:#1a3a5c;color:white;padding:10px 16px;border-radius:6px;margin:18px 0 8px 0"><b style="font-size:1.15em">Part 0 — Environment Setup</b></div>


Run the cell below to install any missing packages. If you are on **Google Colab** this
handles everything. On a local machine with the course conda/venv environment already
activated, the installs will simply be skipped.


In [ ]:
# Install / upgrade required packages (safe to run more than once)

import subprocess, sys
pkgs = ['opencv-python', 'numpy', 'matplotlib', 'scikit-image', 'Pillow', 'requests']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
print('Done.')


Now import everything and confirm the versions.
**Expected:** Python ≥ 3.10, OpenCV ≥ 4.8, NumPy ≥ 1.24.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from skimage import data as skdata
import requests, io, warnings
from PIL import Image

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 11})

print(f'Python  : {sys.version.split()[0]}')
print(f'OpenCV  : {cv2.__version__}')
print(f'NumPy   : {np.__version__}')
import sys; print(f'Python  : {sys.version.split()[0]}')
print('\n✅  All packages loaded successfully!')


<div style="background:#1a3a5c;color:white;padding:10px 16px;border-radius:6px;margin:18px 0 8px 0"><b style="font-size:1.15em">Part 1 — Loading & Displaying Images</b></div>


### How images are loaded

There are several ways to load an image in Python. The three most common in CV work are:

| Method | Returns | Colour order |
|---|---|---|
| `cv2.imread(path)` | NumPy `uint8` array | **BGR** |
| `plt.imread(path)` | NumPy `uint8` or `float32` array | **RGB** |
| `PIL.Image.open(path)` | PIL Image object | **RGB** |

We will primarily use **OpenCV** (`cv2`) in this course because it is the industry standard for
real-time CV pipelines. Just remember: **OpenCV reads images in BGR order**, not RGB — this
trips up almost every beginner. We'll see the consequences in Part 2.

The cell below downloads an aerial image of Prince Edward Island from the public
course repository on GitHub, just to show an example of how to do it. It then loads a
built-in coffee cup image that is used in the examples.

We will revisit aerial images later.

In [ ]:
# ── Load a sample image ────────────────────────────────────────────────────
# We try to grab a real PEI aerial image to show how it can be done

# All course images are served from the public course repository, so they stay
# available for the whole semester.

DATA_BASE = ("https://raw.githubusercontent.com/andrewgodbout/"
             "MCS-3950-F26/main/Notebooks/data/")

# 2010 aerial photo of Panmure Island, Eastern PEI
PEI_URL = DATA_BASE + "pei_panmure_2010.jpg"

def load_from_url(url):
    """Download an image from a URL and return it as an OpenCV BGR array."""
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        arr = np.frombuffer(resp.content, np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)  # returns BGR
        return img
    except Exception as e:
        print(f'Download failed ({e}). Using built-in fallback.')
        return None

#use the above function to load an image from the internet
img_internet_bgr = load_from_url(PEI_URL)

if img_internet_bgr is not None:
    print('Downloaded PEI aerial image successfully!')

#load the coffee cup (note this is RGB format)
img_rgb_fallback = skdata.coffee()


#convert to BGR to simulate cv2 load
img_bgr = cv2.cvtColor(img_rgb_fallback, cv2.COLOR_RGB2BGR)
print('Using built-in coffee image for examples.')


print(f'Image loaded — shape: {img_bgr.shape}, dtype: {img_bgr.dtype}')


### Displaying with Matplotlib

Because Matplotlib expects **RGB** and OpenCV gives us **BGR**, we must convert before
displaying. The function `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` does this swap.

> 🔑 **Rule of thumb:** Always convert BGR → RGB before `plt.imshow()`. You will see *exactly*
> what happens if you forget this in the next section.


In [ ]:
# Convert BGR → RGB for display
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(img_bgr)                 # ← intentionally WRONG (BGR fed to imshow)
axes[0].set_title('❌  Raw BGR (wrong colours!)')
axes[0].axis('off')

axes[1].imshow(img_rgb)                 # ← correct
axes[1].set_title('✅  After BGR→RGB conversion')
axes[1].axis('off')

plt.suptitle('The BGR vs RGB gotcha', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Saving an image

`cv2.imwrite(filename, img_bgr)` saves an image. It expects **BGR** input — since we
loaded with OpenCV the array is already in the correct order.

*Note:* This saves the image in your Files area of the local google drive instance. You can see this by clicking in the sidebar on the file folder icon.

You can then download the image (pei_sample.jpg). Be warned the default drive instance disappears (including files saved there) when you leave or reload this page (and possibly upon a timeout) or kernel restart.

Alternatively you can mount your google drive and save and load images or other files directly from there (in which case they persist between sessions and drops).


In [ ]:
# Save a copy of our image
cv2.imwrite('pei_sample.jpg', img_bgr)
print('Saved pei_sample.jpg')

# Reload it to confirm the round-trip
reloaded = cv2.imread('pei_sample.jpg')
print(f'Reloaded shape: {reloaded.shape}  — matches original: {reloaded.shape == img_bgr.shape}')


<div style="background:#1a3a5c;color:white;padding:10px 16px;border-radius:6px;margin:18px 0 8px 0"><b style="font-size:1.15em">Part 2 — Images as NumPy Arrays</b></div>


A digital image is simply a 3-D NumPy array with shape **(height, width, channels)**.

| Attribute | Meaning | Typical value |
|---|---|---|
| `img.shape` | `(H, W, C)` — rows × columns × channels | `(720, 1280, 3)` |
| `img.dtype` | Data type of each element | `uint8` (0–255) |
| `img.size` | Total number of elements (H × W × C) | e.g. `2764800` |
| `img.nbytes` | Memory footprint in bytes | e.g. `2764800` |

Pixel coordinates use **(row, col)** indexing — the **opposite** order from (x, y):
```
pixel = img[row, col]        # → [B, G, R] values for that pixel
region = img[r1:r2, c1:c2]  # → sub-image (crop)
```


In [ ]:
# ── Inspect the image array ────────────────────────────────────────────────
print('=== Image Array Properties ===')
print(f'  Shape  (H, W, C) : {img_bgr.shape}')
print(f'  Height           : {img_bgr.shape[0]} pixels')
print(f'  Width            : {img_bgr.shape[1]} pixels')
print(f'  Channels         : {img_bgr.shape[2]}')
print(f'  dtype            : {img_bgr.dtype}')
print(f'  Min pixel value  : {img_bgr.min()}')
print(f'  Max pixel value  : {img_bgr.max()}')
print(f'  Mean pixel value : {img_bgr.mean():.1f}')
print(f'  Memory           : {img_bgr.nbytes / 1e6:.2f} MB')


In [ ]:
# ── Access individual pixels ───────────────────────────────────────────────
# Pick a pixel near the centre of the image
cy, cx = img_bgr.shape[0] // 2, img_bgr.shape[1] // 2
pixel_bgr = img_bgr[cy, cx]          # OpenCV stores B, G, R
pixel_rgb = img_rgb[cy, cx]          # RGB order

print(f'Centre pixel (row={cy}, col={cx})')
print(f'  BGR values : B={pixel_bgr[0]}, G={pixel_bgr[1]}, R={pixel_bgr[2]}')
print(f'  RGB values : R={pixel_rgb[0]}, G={pixel_rgb[1]}, B={pixel_rgb[2]}')

# Mark it on the image
marker = img_rgb.copy()
cv2.drawMarker(marker, (cx, cy), (255, 0, 0), cv2.MARKER_CROSS, 40, 3)

plt.figure(figsize=(6, 4))
plt.imshow(marker)
plt.title(f'Centre pixel  R={pixel_rgb[0]}  G={pixel_rgb[1]}  B={pixel_rgb[2]}')
plt.axis('off')
plt.tight_layout()
plt.show()


<div style="background:#1a3a5c;color:white;padding:10px 16px;border-radius:6px;margin:18px 0 8px 0"><b style="font-size:1.15em">Part 3 — Colour Channels & Colour Spaces</b></div>


### Splitting into R, G, B channels

A colour image is the *combination* of three single-channel (grayscale) images — one per
colour. Splitting them reveals what information each channel carries:

- **Red channel** — highlights warm surfaces (sand, red PEI soil)
- **Green channel** — highlights vegetation
- **Blue channel** — highlights water and sky


*Note:* In the below plots we use a colour map (`cmap=...`) to facilitate the plotting. In these particular colour maps the Reds, Greens or Blues appear deep and dark when the image is intensive in that colour at that pixel (and light, white and washed out when that colour is lacking). Typically colours are light and bright for intense colours (and other colour maps).

If we didn't specify a colour map the images would appear with the default colour map (`viridis`) and might look a bit *funky* since it tries to map the single channel onto a gradient colours.

In [ ]:
# cv2.split returns channels in B, G, R order
b, g, r = cv2.split(img_bgr)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
ax = axes.ravel()


#note that these color maps work a bit backwards: low/nonexistent red is mapped to white (in cmap='Reds')
#while intense red is mapped to deep dark Red (and similar for green or blue)
ax[0].imshow(img_rgb_fallback);           ax[0].set_title('Original (RGB)'); ax[0].axis('off')
ax[1].imshow(r, cmap='Reds');    ax[1].set_title('Red channel');    ax[1].axis('off')
ax[2].imshow(g, cmap='Greens');  ax[2].set_title('Green channel');  ax[2].axis('off')
ax[3].imshow(b, cmap='Blues');   ax[3].set_title('Blue channel');   ax[3].axis('off')

plt.suptitle('RGB Channel Decomposition', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Each channel shape:', r.shape, '← same H×W, but single channel (2-D array)')


### Converting to Grayscale

Grayscale reduces a 3-channel image to 1 channel. OpenCV uses the weighted formula:

$$Y = 0.114 \cdot B + 0.587 \cdot G + 0.299 \cdot R$$

The weights reflect human visual sensitivity (we are most sensitive to green light).

> ⚠️ Note that a simple average of R, G, B gives a different (less perceptually accurate) result.


In [ ]:
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
gray_naive = np.mean(img_bgr, axis=2).astype(np.uint8)  # simple average

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img_rgb);                    axes[0].set_title('Colour (RGB)')
axes[1].imshow(gray, cmap='gray');          axes[1].set_title('Grayscale (OpenCV weighted)')
axes[2].imshow(gray_naive, cmap='gray');    axes[2].set_title('Naive average (R+G+B)/3')
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

print(f'Grayscale array shape: {gray.shape}   ← 2-D now (no channel axis)')


In [ ]:
#in the above example (assuming coffee cup) the values mostly differ in the actual coffee.
#It might be hard to see but can be more extreme with other more colorful images

#diff the images to capture the absolute differences
diff = cv2.absdiff(gray, gray_naive)

#threshold the differences so that anything 10 or above (larger difference) is set to 255 (white)
_, thresholded_diff = cv2.threshold(diff, 10, 255, cv2.THRESH_BINARY)
plt.imshow(thresholded_diff, cmap='gray')



### HSV Colour Space

**HSV** (Hue, Saturation, Value) separates *what colour* (hue) from *how vivid* (saturation)
and *how bright* (value). This makes it much easier to isolate colours by hue range, which
is useful for tasks like detecting the red PEI soil, green farmland, or blue water.

| Channel | Range in OpenCV | Meaning |
|---|---|---|
| H (Hue) | 0 – 179 | Colour angle on the colour wheel |
| S (Saturation) | 0 – 255 | 0 = grey, 255 = fully vivid colour |
| V (Value) | 0 – 255 | 0 = black, 255 = full brightness |


In [ ]:
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
h, s, v = cv2.split(hsv)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(img_rgb);          axes[0].set_title('Original RGB')
axes[1].imshow(h, cmap='hsv');    axes[1].set_title('H — Hue')
axes[2].imshow(s, cmap='gray');   axes[2].set_title('S — Saturation')
axes[3].imshow(v, cmap='gray');   axes[3].set_title('V — Value (brightness)')
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()


<div style="background:#1a3a5c;color:white;padding:10px 16px;border-radius:6px;margin:18px 0 8px 0"><b style="font-size:1.15em">Part 4 — Basic Spatial Operations</b></div>


### Cropping

Cropping is simply **array slicing**: `img[row_start:row_end, col_start:col_end]`.
No function call needed — this is one of the reasons NumPy is so powerful for image work.


In [ ]:
H, W = img_rgb.shape[:2]

# Crop the top right portion
crop = img_rgb[0 : H//2,  W//2 : ]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(img_rgb)
# Draw the crop rectangle on the original
rect = plt.Rectangle((W//2, 0), W//2, H//2,
                      edgecolor='red', linewidth=2, facecolor='none')
axes[0].add_patch(rect)
axes[0].set_title('Original — red box = crop region')
axes[0].axis('off')

axes[1].imshow(crop)
axes[1].set_title(f'Cropped region  {crop.shape[1]}×{crop.shape[0]} px')
axes[1].axis('off')

plt.tight_layout(); plt.show()


### Resizing

`cv2.resize(img, (new_width, new_height), interpolation=...)` — note that OpenCV
takes **(width, height)** not (height, width). The interpolation method matters:

| Method | Use case |
|---|---|
| `cv2.INTER_AREA` | Shrinking (downsampling) — avoids moiré |
| `cv2.INTER_LINEAR` | General purpose (fast) |
| `cv2.INTER_CUBIC` | Enlarging — smoother than linear |


In [ ]:
scale_factors = [0.25, 0.5, 1.0, 2.0]

# Set height ratios based directly on scale factors
fig, axes = plt.subplots(
    4, 1,
    figsize=(8, 12),
    gridspec_kw={'height_ratios': scale_factors}
)

for ax, s in zip(axes, scale_factors):
    #new width and new height
    nw, nh = int(W * s), int(H * s)

    #shrink or enlarge?
    interp = cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC
    resized = cv2.resize(img_rgb, (nw, nh), interpolation=interp)

    ax.imshow(resized)
    ax.set_title(f'{s}× → {nw}×{nh}', fontsize=10)
    ax.axis('off')

plt.suptitle('Resizing with dynamic subplot height ratios', fontweight='bold')
plt.tight_layout()
plt.show()

### Flipping and Rotating


In [ ]:
flip_h  = cv2.flip(img_rgb, 1)   # flipCode=1 → horizontal
flip_v  = cv2.flip(img_rgb, 0)   # flipCode=0 → vertical

# Rotate 45° around image centre (keeping full image)
centre = (W // 2, H // 2)
M = cv2.getRotationMatrix2D(centre, angle=45, scale=1.0)
rotated = cv2.warpAffine(img_rgb, M, (W, H),
                          borderMode=cv2.BORDER_REFLECT)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, im, title in zip(axes,
    [img_rgb, flip_h, flip_v, rotated],
    ['Original', 'Flip horizontal', 'Flip vertical', 'Rotate 45°']):
    ax.imshow(im); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()


<div style="background:#1a3a5c;color:white;padding:10px 16px;border-radius:6px;margin:18px 0 8px 0"><b style="font-size:1.15em">Part 5 — Image Histograms</b></div>


A histogram counts how many pixels have each intensity value (0–255). It tells you
about the **tonal distribution** of an image:

- Peaks toward 0 → image is dark
- Peaks toward 255 → image is bright / overexposed
- Wide, flat distribution → high contrast
- Narrow peak → low contrast

Histograms are used in **histogram equalisation**, **thresholding**, and
**image retrieval**.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: the image
axes[0].imshow(img_rgb); axes[0].set_title('Image'); axes[0].axis('off')

# Right: per-channel histogram
channel_info = [('Red', r, 'red'), ('Green', g, 'green'), ('Blue', b, 'blue')]
for name, channel, colour in channel_info:
    hist = cv2.calcHist([channel], [0], None, [256], [0, 256])
    axes[1].plot(hist, color=colour, linewidth=1.2, label=name, alpha=0.8)

axes[1].set_xlim([0, 255])
axes[1].set_xlabel('Pixel intensity')
axes[1].set_ylabel('Pixel count')
axes[1].set_title('RGB Histogram')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


In [ ]:
# ── Histogram equalisation on the grayscale image ──────────────────────────
# This stretches the histogram to use the full 0-255 range → improves contrast.
gray_eq = cv2.equalizeHist(gray)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

axes[0,0].imshow(gray,    cmap='gray'); axes[0,0].set_title('Original grayscale'); axes[0,0].axis('off')
axes[0,1].imshow(gray_eq, cmap='gray'); axes[0,1].set_title('After equalisation');  axes[0,1].axis('off')

for ax, img_ch, title in [
    (axes[1,0], gray,    'Histogram — original'),
    (axes[1,1], gray_eq, 'Histogram — equalised'),
]:
    hist = cv2.calcHist([img_ch], [0], None, [256], [0,256])
    ax.fill_between(range(256), hist.flatten(), alpha=0.6, color='steelblue')
    ax.set_xlim([0, 255]); ax.set_title(title)
    ax.set_xlabel('Intensity'); ax.set_ylabel('Count')
    ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()


<div style="background:#2e6da4;color:white;padding:10px 16px;border-radius:6px;margin:18px 0 8px 0"><b style="font-size:1.15em">Part 6 — Exercises</b></div>


> Complete **all four exercises** below. Each exercise has a `# YOUR CODE HERE` marker or similar indication to fill in your solution
> You should not need more than ~10 lines of code per exercise.
> Include a brief written answer (1–3 sentences) in the markdown cell that follows each exercise.


### Exercise 1 — Load and Inspect an Image

In the below *url* variable there is an aerial photo from 2020 of Eastern PEI. It shows some forest, farm fields, coast line and buildings.

A. Load the image by calling the load_from_url method defined earlier in this notebook.

B. Display the image dimensions, data type and the max and min pixel values.

C. Convert the image to RGB and then display the image.


In [ ]:
# ── Exercise 1 ─────────────────────────────────────────────────────────────
# 1a. Load an image
# url points to a 2020 aerial photo of Eastern PEI: forest, farm fields,
# coastline and buildings. It is served from the public course repository.
url = DATA_BASE + "eastern_pei_aerial_2020.png"

# Load the image from url
my_img_bgr = ...# YOUR CODE HERE  -- hint: load_from_url(url)

In [ ]:
# 1b. Print height, width, number of channels, dtype, and min/max pixel values

# YOUR CODE HERE
#height, width, number of channels

# dtype

#min and max pixel values

#1c. convert the image to RGB

#Display the image

#adjust the display size
plt.figure(figsize=(10,10))
#EDIT the below line
plt.imshow('''your_rgb_image_variable_name''')

#note if you do not want the axis
#plt.axis('off')
plt.show()



**Your answer:** *(describe what the image shows and note one interesting thing
you observe about its pixel statistics)*

*Write your answer here.*


### Exercise 2 — Colour Channel Analysis

For your image from Exercise 1:
1. Split into R, G, B channels (remember that cv2.split wants the image in B,G,R order).
2. Display all three channels side-by-side using a coloured colourmap (e.g. `cmap='Reds'`).
3. Compute the **mean** value of each channel and print them.
4. Which channel has the highest mean? What does that tell you about the image?


In [ ]:
# ── Exercise 2 ─────────────────────────────────────────────────────────────
# 2a. Split channels
# YOUR CODE HERE


# 2b. Display side by side
fig, axes = plt.subplots(1,4, figsize=(16,8))
# YOUR CODE HERE
plt.show()

# 2c. Print mean of each channel
# YOUR CODE HERE




**Your answer:** *(which channel dominates and why does that make sense for your image?)*

*Write your answer here.*


### Exercise 3 — Region of Interest (ROI) Extraction

You are analysing an aerial image of eastern PEI and want to focus on a
specific sub-region to measure its spectral properties.

The photo shows Panmure Island, notice that in the bottom middle of the image is a farm field with a laneway on its right side, that narrows at the bottom and leads to the beach (and a few houses along the shore further to the right). To the left of this field at the coast is a Beech Tree forest (deciduous trees, broadleaf, hardwood), to the right of the field near the coast / houses is a Spruce Tree force (coniferous trees, needles, softwood).

1. Define a region of interest that roughly captures the two types of forest in a single image.

**Note:** This involves a bit of visual estimation, *hint: try around x: 600–1000, y: 1050–1350, then refine*

2. Display the original image with a coloured rectangle drawn around the ROI side-by-side with an image of the extracted ROI. Your figure should roughly match the below:
![forests of panmure island](https://raw.githubusercontent.com/andrewgodbout/MCS-3950-F26/main/Notebooks/data/target.png "Figure of Panmure Island")

4. Create 2 more extracted regions of interest, 1: containing the Beech trees and 2: containing the Spruce Trees similar to below:
![crop of beech and spruce](https://raw.githubusercontent.com/andrewgodbout/MCS-3950-F26/main/Notebooks/data/bs.png "Beech and Spruce Trees")

**Please** ensure your cropped beech and cropped spruce images are saved in RGB version into variables called `my_crop_beech_rgb` and `my_crop_spruce_rgb` respectively


In [ ]:
# ── Exercise 3 ─────────────────────────────────────────────────────────────
img_copy = img_rgb.copy()

# 3.1 Define ROI coordinates
# YOUR CODE HERE
x1 = ...
x2 = ...
y1 = ...
y2 = ...



# 3.2 Draw rectangle on a copy of the image (use cv2.rectangle)
# Hint: cv2.rectangle(img_copy, (x1, y1), (x2, y2), (R, G, B), thickness)
# Note: cv2.rectangle uses (col, row) = (x, y) order!
# R,G,B is the color of the rectangle ex. (255,0, 0) for red!
# YOUR CODE HERE


# 3.2 Crop the ROI into a separate image using a slice
# YOUR CODE HERE


# 3.2 Display the original (with red bounding rectangle) and cropped ROI side by side (don't forget to convert to RGB for display)

#subfigure for 2 images side by side
fig, axes = plt.subplots(1,2, figsize=(10,10))

# YOUR CODE HERE

plt.show()

# 3.3 Extract the two new ROI with Beech trees and Spruce Trees




# 3.3 Display the Beech and Spruce Tree Images, make sure your variables are called: my_crop_beech_rgb and my_crop_spruce_rgb

# placeholders for now (remove the below but retain the variable names with your solution)
my_crop_beech_rgb = img_copy
my_crop_spruce_rgb = img_copy


**Question:** *Describe the differences between the Beech trees and Spruce Trees*?

*Write your answer here.*


### Exercise 4 — Computing Pixel Color Differences between Beech and Spruce

Visually it looks like there is a difference in the *green* color between beech and spruce (beech with lighter green and spruce with darker-denser green). In this exercise we compute 3 RGB based vegetation indices to see if any are suitable for separating beech and spruce trees for the purpose of identification.

The three indices are:

1. **Triangular Greeness Index (TGI)**: G - 0.39R - 0.61B (approximates spectral band formed by RGB bands ... perhaps useful for leaf pigment)
2. **Normalized Green-Red Difference Index (NGRDI)**: $\frac {G-R}{G+R}$ (ignores the blue channel that might be associated with fog or shadows)
3. **Excess Green (ExG)**: 2G - R - B (amplify the green by double weighting it - sensitive to brightness)

We will apply the above formulas at each pixel to augment the image of the beech and spruce trees to see if we can more easily classify a group of pixels based on their coloring.

Steps to complete exercise 4:

1. convert the image to float32 with values in the range [0,1] (implement: `prepare_float_rgb`)
2. separate out the colour channels (`splitRGB`)
3. implement the formulas (no loops or special operations, just operate on the whole colour channel array)
4. print the mean for each index


Once you've completed the applicable functions you should be able to see a histogram and heat map of the 3 potential indices.


In [ ]:
# ── Exercise 4 ─────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import cv2

# 4.1
def prepare_float_rgb(img):
    '''convert image to 32 bit float version with values in the range [0.0, 1.0]
       img: unit8 image
       return float32 normalized image
    '''

    # TODO replace the below with your code
    return img

# 4.2
def splitRGB(fimg):
    ''' Split a floating point 32 RGB image into separate R, G and B channels
    '''

    # TODO replace the below with your code
    return fimg,fimg,fimg


# 4.3
def TGI(R, G, B):
    ''' calculate and return the TGI value for the image at each pixel given the R,G,B channels

        R: the [0.0-1.0] values of the red channel of the image
        G: the [0.0-1.0] values of the green channel of the image
        B: the [0.0-1.0] values of the blue channel of the image

        return the Triangular Greeness Index for the R, G, B image
    '''

    # TODO replace the below with your code
    return G


def NGRDI(R, G, B):
    ''' calculate and return the NGRDI value for the image at each pixel given the R,G,B channels

        R: the [0.0-1.0] values of the red channel of the image
        G: the [0.0-1.0] values of the green channel of the image
        B: the [0.0-1.0] values of the blue channel of the image

        Note: avoid possible divide by zero

        return the Normalized Green-Red Difference Index for the R, G, B image
    '''

    #TODO replace the below with your code
    return G

def ExG(R, G, B):
    ''' calculate and return the ExG value for the image at each pixel given the R,G,B channels

        R: the [0.0-1.0] values of the red channel of the image
        G: the [0.0-1.0] values of the green channel of the image
        B: the [0.0-1.0] values of the blue channel of the image

        Note: avoid possible divide by zero

        return the Excess Green values for the R, G, B image
    '''

    # TODO replace the below with your code
    return G

#4.4
def print_means(tgi, ngrdi, exg):
    ''' print the mean value for each of the 3 indices

    tgi: an np array of TGI values
    ngrdi: an np array of NGRDI values
    exg: an np array of ExG values

    return nothing (just print)

    '''

    # TODO Add your code here



def plot_histograms(beech_rgb, spruce_rgb, num_bins=40):
    '''
        run the calculations and then plot the augmented images (heat map) and histograms

        beech_rgb: an RGB image containing mostly beech trees
        spruce_rgb: an RGB image containing mostly spruce trees
        num_bins: the bins in the histogram

        You should not have to modify the below code. It will call the methods you have created above.

    '''
    #convert to float
    beech_flt_rgb = prepare_float_rgb(beech_rgb)
    spruce_flt_rgb = prepare_float_rgb(spruce_rgb)

    #split channels
    bR, bG, bB = splitRGB(beech_flt_rgb)
    sR, sG, sB = splitRGB(spruce_flt_rgb)

    #compute indices
    beech_tgi = TGI(bR, bG, bB)
    beech_ngrdi = NGRDI(bR, bG, bB)
    beech_exg = ExG(bR, bG, bB)

    spruce_tgi = TGI(sR,sG, sB)
    spruce_ngrdi = NGRDI(sR, sG, sB)
    spruce_exg = ExG(sR, sG, sB)


    beech_values = {}
    beech_values["TGI"] = beech_tgi
    beech_values["NGRDI"] = beech_ngrdi
    beech_values["ExG"] = beech_exg

    spruce_values = {}
    spruce_values["TGI"] = spruce_tgi
    spruce_values["NGRDI"] = spruce_ngrdi
    spruce_values["ExG"] = spruce_exg

    #print beech means
    print("Means for Beech indices")
    print_means(beech_tgi, beech_ngrdi, beech_exg)

    #print spruce means
    print("Means for Spruce indices")
    print_means(spruce_tgi, spruce_ngrdi, spruce_exg)

    indices = ["TGI", "NGRDI", "ExG"]
    cmaps = {
        "ExG": "Greens",
        "TGI": "Greens",
        "NGRDI": "Greens"
    }

    # Set up a 3x3 figure layout
    fig = plt.figure(figsize=(16, 18), dpi=100)
    gs = fig.add_gridspec(3, 3, width_ratios=[1, 1, 1.2])

    #do plotting
    for i, k in enumerate(indices):

        beech_pixels = beech_values[k]
        spruce_pixels = spruce_values[k]

        val_min = float(min(beech_pixels.min(), spruce_pixels.min()))
        val_max = float(max(beech_pixels.max(), spruce_pixels.max()))

        # 1. Beech Heatmap
        ax_beech = fig.add_subplot(gs[i, 0])
        im1 = ax_beech.imshow(beech_pixels, cmap=cmaps[k], vmin=val_min, vmax=val_max)
        ax_beech.set_title(f"Beech - {k}", fontsize=11, fontweight='bold')
        ax_beech.axis("off")
        plt.colorbar(im1, ax=ax_beech, fraction=0.046, pad=0.04)

        # 2. Spruce Heatmap
        ax_spruce = fig.add_subplot(gs[i, 1])
        im2 = ax_spruce.imshow(spruce_pixels, cmap=cmaps[k], vmin=val_min, vmax=val_max)
        ax_spruce.set_title(f"Spruce - {k}", fontsize=11, fontweight='bold')
        ax_spruce.axis("off")
        plt.colorbar(im2, ax=ax_spruce, fraction=0.046, pad=0.04)

        #histograms
        beech_hist = cv2.calcHist([beech_pixels], [0], None, [num_bins], [val_min, val_max])
        spruce_hist = cv2.calcHist([spruce_pixels], [0], None, [num_bins], [val_min, val_max])

        bin_edges = np.linspace(val_min, val_max, num_bins + 1)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        # Normalize histograms (Density PDF)
        bin_width = (val_max - val_min) / num_bins
        beech_hist_density = beech_hist / (beech_hist.sum() * bin_width + 1e-8)
        spruce_hist_density = spruce_hist / (spruce_hist.sum() * bin_width + 1e-8)

        ax_dist = fig.add_subplot(gs[i, 2])
        ax_dist.plot(bin_centers, beech_hist_density, color="forestgreen", linewidth=1.2, label="beech")
        ax_dist.fill_between(bin_centers, beech_hist_density.ravel(), color="mediumseagreen", alpha=0.3)

        ax_dist.plot(bin_centers, spruce_hist_density, color="black", linewidth=1.2, label="spruce")
        ax_dist.fill_between(bin_centers, spruce_hist_density.ravel(), color="slategray", alpha=0.3)


        # Means vertical lines
        beech_mean = beech_pixels.mean()
        spruce_mean = spruce_pixels.mean()

        ax_dist.axvline(beech_mean, color="forestgreen", linestyle="--", linewidth=1.5, label=f"Beech Mean ({beech_mean:.4f})")
        ax_dist.axvline(spruce_mean, color="black", linestyle="--", linewidth=1.5, label=f"Spruce Mean ({spruce_mean:.4f})")


        ax_dist.set_title(f"{k} Density Histogram", fontsize=11, fontweight='bold')
        ax_dist.set_xlabel(f"{k} Value")
        ax_dist.set_ylabel("Density")
        ax_dist.legend(loc="upper right", fontsize=8)
        ax_dist.grid(True, linestyle=":", alpha=0.6)

    plt.tight_layout()
    plt.show()


# Do the calculations and plot
plot_histograms(my_crop_beech_rgb, my_crop_spruce_rgb)

**Question:** *Which metric(s) might be appropriate for identifying Beech from Spruce? Why?*

*Write your answer here.*


### Notes on Exercise 4

We used RGB images for exercise 4 but that might not be the best color model. Next week we will learn about alternative color models that may be better suited to inherently handle challenging issues such as brightness changes.

This exercise ties in with the forest related project and especially how you might build the labels for beech and spruce (or hardwood and softwood) using the richer color dataset and then migrating those labels to grayscale.


### Notes on the Active Learning

For those who did not finish in class time please complete the exercises before next class and ask any questions at the start of next class.

<div style="background:#1e7e34;color:white;padding:10px 16px;border-radius:6px;margin:18px 0 8px 0"><b style="font-size:1.15em">End of Active Learning 1</b></div>
